# Практическая работа. Снижение размерности: метод главных компонент
## «Сжать 13 признаков до двух и почти ничего не потерять»

**Цель:** увидеть на опыте проклятие размерности, написать на чистом Python метод главных компонент (центрирование, ковариационная матрица, степенной метод, исчерпывание, проекция и восстановление), спроецировать набор Wine из 13 признаков на плоскость, честно сравнить метод k ближайших соседей до и после снижения размерности через кросс-валидацию и понять, когда метод главных компонент не помогает.

> **Правило работы.** Внутри своих функций – только списки, циклы, `math` и `random`. Библиотеки `sklearn` и `numpy` нужны только для загрузки наборов данных и сверки, `matplotlib` – только в готовых ячейках с графиками.

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом – подсказка написана в комментарии `# <--`. Напечатанные значения должны совпасть с тем, что стоит после слова «ожидается». Готовые ячейки просто выполните. Сначала прочитайте теорию.

**План работы**

| Задание | Что пишем | Главный результат |
|---|---|---|
| П1 | `near_far_ratio`, `shell_share`, случайные признаки | в пространстве большой размерности все точки почти одинаково далеко |
| П2 | `mean_vector`, `center`, `covariance` | ковариационная матрица облака точек машины |
| П3 | `matvec`, `norm`, `power_iteration` | первая компонента показывает, куда повернута машина |
| П4 | `deflate`, `pca_fit`, `pca_transform` | вторая компонента и доля объясненной дисперсии |
| П5 | Wine на плоскости | две компоненты из 13 хранят 55 % дисперсии, сорта видны глазами |
| П6 | `pca_inverse`, ошибка восстановления | ошибка равна сумме отброшенных дисперсий |
| П7 | метод k ближайших соседей с главными компонентами в кросс-валидации | 2 признака вместо 13 почти без потери точности |
| П8 | когда метод не помогает | шумовые признаки: компоненты находят шум, а не пользу |

## Теория: снижение размерности и метод главных компонент

**1. Зачем снижать размерность.** Каждый признак объекта – одна ось пространства. У объектов на дороге из Б1.2 было 4 признака, у вина – 13, у изображения рукописной цифры 8×8 – 64 (яркость каждого пикселя), у кадра камеры беспилотного автомобиля – миллионы. Снижение размерности – переход от $d$ признаков к $k < d$ новым признакам так, чтобы потерять как можно меньше информации. Это нужно, чтобы:

- **увидеть** данные: 13-мерное облако нельзя нарисовать, а двумерное можно;
- **ускорить** модели: метод k ближайших соседей считает расстояния по всем признакам;
- **сжать** данные и убрать мелкий шум;
- **ослабить проклятие размерности.**

**2. Проклятие размерности.** Так называют набор эффектов, из-за которых в пространстве большой размерности перестает работать интуиция, привычная по плоскости.

- *Расстояния выравниваются.* Возьмем случайные точки в кубе $[0, 1]^d$ и для новой точки найдем отношение расстояния до ближайшей точки к расстоянию до самой далекой. На прямой оно почти 0: ближайший сосед очень близко, самый далекий очень далеко. При $d = 1000$ оно около 0,9: все точки почти одинаково далеко. Для метода k ближайших соседей это беда: «ближайшие» соседи перестают быть по-настоящему близкими.
- *Почти весь объем – у границы.* Доля объема куба $[0, 1]^d$ в слое толщиной 0,05 у каждой грани равна $1 - 0{,}9^d$: при $d = 2$ это 19 %, при $d = 50$ уже 99,5 %. Почти все точки оказываются «на краю».
- *Нужно очень много данных.* Чтобы покрыть отрезок точками с шагом 0,1, нужно 10 точек, квадрат – 100, куб размерности $d$ – $10^d$.

Отсюда практический вывод: каждый лишний признак, который не несет информации об ответе, ухудшает метод k ближайших соседей.

**3. Среднее, центрирование, дисперсия.** Пусть $X$ – таблица из $n$ объектов и $d$ признаков, $x_{ij}$ – значение признака $j$ у объекта $i$. Вектор средних $\mu_j = \frac{1}{n}\sum_i x_{ij}$. **Центрирование** – вычитание среднего из каждого признака: $z_{ij} = x_{ij} - \mu_j$; после него облако точек «стоит» в начале координат. Дисперсия признака $j$ – средний квадрат отклонения: $\sigma_j^2 = \frac{1}{n}\sum_i z_{ij}^2$.

**4. Ковариация и ковариационная матрица.** Ковариация признаков $j$ и $l$:

$$c_{jl} = \frac{1}{n}\sum_{i=1}^{n} z_{ij}\, z_{il}.$$

Если признаки растут вместе, произведения отклонений в основном положительны и ковариация больше нуля; если один растет, а другой падает – меньше нуля; если связи нет – около нуля. Ковариационная матрица $C$ размера $d \times d$ собирает все $c_{jl}$: на диагонали стоят дисперсии ($c_{jj} = \sigma_j^2$), матрица симметрична ($c_{jl} = c_{lj}$). Сумма диагонали $c_{11} + \dots + c_{dd}$ – **общая дисперсия** облака.

*Замечание.* Библиотеки `numpy` и `sklearn` при расчете дисперсии и ковариации обычно делят на $n - 1$, а не на $n$. Все дисперсии от этого умножаются на одно и то же число $\frac{n}{n-1}$, поэтому направления компонент и доли дисперсии не меняются. Мы делим на $n$, как при стандартизации в Б1.6.

**5. Дисперсия вдоль направления.** Направление в пространстве задается единичным вектором $v$ ($\|v\| = \sqrt{v_1^2 + \dots + v_d^2} = 1$). **Проекция** центрированного объекта $z$ на это направление – одно число, скалярное произведение:

$$t = z \cdot v = z_1 v_1 + \dots + z_d v_d.$$

Это координата «тени» точки на прямой, проходящей через центр облака в направлении $v$. Дисперсия проекций всех объектов равна $\frac{1}{n}\sum_i (z_i \cdot v)^2 = v^{\top} C v$. Значит, ковариационная матрица позволяет посчитать разброс вдоль любого направления, не проецируя точки.

**6. Главные компоненты.** **Первая главная компонента** $v_1$ – направление, вдоль которого дисперсия проекций $v^{\top} C v$ наибольшая. Вдоль него облако сильнее всего «вытянуто», поэтому одна координата $t_1 = z \cdot v_1$ сохраняет о точке больше всего информации. **Вторая главная компонента** $v_2$ – направление наибольшей дисперсии среди перпендикулярных $v_1$, третья перпендикулярна первым двум, и так далее. Всего компонент $d$; вместе они образуют новую систему координат, повернутую по облаку точек.

Из линейной алгебры известно, что главные компоненты – это **собственные векторы** матрицы $C$: $C v = \lambda v$, то есть умножение на матрицу только растягивает вектор в $\lambda$ раз, не поворачивая его. Число $\lambda$ (**собственное значение**) равно дисперсии вдоль этой компоненты: $v^{\top} C v = v^{\top}\lambda v = \lambda$. Компоненты нумеруют по убыванию: $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_d \ge 0$, а их сумма равна общей дисперсии.

**7. Доля объясненной дисперсии.** Компонента $m$ «объясняет» долю $\dfrac{\lambda_m}{\lambda_1 + \dots + \lambda_d}$ общей дисперсии. Сумма долей первых $k$ компонент показывает, сколько разброса данных сохранится, если оставить только $k$ новых признаков. Число компонент выбирают, например, так, чтобы сохранить 90 % дисперсии, или по кросс-валидации модели, которая работает после снижения размерности.

**8. Степенной метод.** Как найти $v_1$ без готовых библиотек? Возьмем случайный вектор $v$ и будем много раз умножать его на $C$, каждый раз приводя к длине 1:

$$v \leftarrow \frac{C v}{\|C v\|}.$$

Разложим начальный вектор по компонентам: $v = a_1 v_1 + a_2 v_2 + \dots$. После $t$ умножений получится $a_1 \lambda_1^t v_1 + a_2 \lambda_2^t v_2 + \dots$. Так как $\lambda_1$ больше остальных, первое слагаемое растет быстрее всех и после нормировки остается только $v_1$. Ошибка убывает примерно как $(\lambda_2/\lambda_1)^t$: чем больше разница между $\lambda_1$ и $\lambda_2$, тем быстрее сходимость. Остановимся, когда вектор почти перестанет меняться. Дисперсия вдоль найденного направления $\lambda_1 = v^{\top} C v$.

Векторы $v_1$ и $-v_1$ задают одну и ту же прямую. Чтобы ответ был однозначным, договоримся: **наибольшая по модулю координата вектора положительна**.

**9. Исчерпывание.** Чтобы найти вторую компоненту, «вычтем» из матрицы найденную:

$$C' = C - \lambda_1 v_1 v_1^{\top}, \qquad c'_{jl} = c_{jl} - \lambda_1 v_{1j} v_{1l}.$$

У новой матрицы вдоль $v_1$ дисперсия стала нулевой, а остальные компоненты остались прежними. Значит, степенной метод для $C'$ найдет $v_2$. Повторяя, находим столько компонент, сколько нужно.

**10. Преобразование и восстановление.** Метод главных компонент с $k$ компонентами:

- **обучение:** по обучающей выборке найти средние $\mu$ и компоненты $v_1, \dots, v_k$;
- **преобразование:** новый объект $x$ переходит в $k$ чисел $t_m = (x - \mu)\cdot v_m$;
- **восстановление:** обратно $\hat{x} = \mu + t_1 v_1 + \dots + t_k v_k$ – точка, ближайшая к $x$ среди лежащих в пространстве первых $k$ компонент.

Средний квадрат ошибки восстановления $\frac{1}{n}\sum_i \|x_i - \hat{x}_i\|^2$ на обучающей выборке ровно равен сумме **отброшенных** дисперсий $\lambda_{k+1} + \dots + \lambda_d$. Главные компоненты – лучшие $k$ направлений именно в этом смысле: никакие другие $k$ направлений не дают меньшей ошибки восстановления.

**11. Важные правила.**

- **Масштаб.** Метод ищет наибольшую дисперсию, а дисперсия зависит от единиц измерения. Признак в тысячах (пролин в Wine) задавит признаки в долях единицы. Поэтому признаки сначала стандартизуют.
- **Только обучающая выборка.** Средние, стандартные отклонения и компоненты считаются по обучающей выборке и затем применяются к валидационной и тестовой, как масштабирование в Б1.6 и Б2.1. Внутри кросс-валидации – заново в каждом блоке.
- **Метод не видит ответов.** Он сохраняет направления с большим разбросом, а не направления, полезные для классификации. Если сильный разброс дает шум, компоненты найдут шум.

**Где почитать.** Я. Гудфеллоу, И. Бенджио, А. Курвилль «Глубокое обучение»: раздел 2.12 «Пример: метод главных компонент», раздел 5.8.1 «Метод главных компонент», раздел 5.11.1 «Проклятие размерности». Открытый курс машинного обучения ODS (mlcourse.ai), тема 7 «Обучение без учителя: PCA и кластеризация» (статья на Хабре). Документация scikit-learn, раздел «Decomposing signals in components», подраздел «Principal component analysis».

In [ ]:
# Готовая ячейка: библиотеки, данные и функции из прошлых работ
import math
import random
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine     # только для загрузки набора Wine

RANDOM_STATE = 42
BLUE, NAVY, GREY, DARK_BLUE = '#4C9BFF', '#0F1B2D', '#969CA3', '#1F63C9'
CLASS_COLORS = [BLUE, NAVY, GREY]

def train_test_split(X, y, test_size=0.3, seed=RANDOM_STATE):
    """Перемешивает объекты и делит их на обучающую и тестовую части (из Б1.2)."""
    idx = list(range(len(X)))
    random.Random(seed).shuffle(idx)
    n_test = int(len(X) * test_size)
    test, train = idx[:n_test], idx[n_test:]
    return ([X[i] for i in train], [X[i] for i in test],
            [y[i] for i in train], [y[i] for i in test])

def make_road_objects(n_per_class=100, seed=RANDOM_STATE):
    """Объекты на дороге (из Б1.2): 0 – пешеход, 1 – велосипедист, 2 – автомобиль.
    Признаки: [скорость, км/ч;  длина, м;  высота, м;  расстояние до объекта, м]."""
    rnd = random.Random(seed)
    params = {0: ((5, 2.5), (0.7, 0.4), (1.7, 0.15)),
              1: ((14, 6), (1.7, 0.45), (1.75, 0.15)),
              2: ((18, 12), (3.6, 1.1), (1.6, 0.2))}
    X, y = [], []
    for c, (pv, pl, ph) in params.items():
        for _ in range(n_per_class):
            v = max(0.5, rnd.gauss(*pv))
            L = max(0.2, rnd.gauss(*pl))
            H = max(0.8, rnd.gauss(*ph))
            D = rnd.uniform(5, 80)
            X.append([round(v, 1), round(L, 2), round(H, 2), round(D, 1)])
            y.append(c)
    idx = list(range(len(X)))
    rnd.shuffle(idx)
    return [X[i] for i in idx], [y[i] for i in idx]

def make_car_cloud(n=200, angle_deg=30, length=4.5, width=1.8, cx=10.0, cy=5.0, seed=RANDOM_STATE):
    """Точки лидара на кузове машины, вид сверху: прямоугольник длиной 4,5 м и шириной 1,8 м,
    повернутый на angle_deg градусов, с центром в точке (cx, cy). Координаты в метрах."""
    rnd = random.Random(seed)
    a = math.radians(angle_deg)
    X = []
    for _ in range(n):
        u, w = rnd.uniform(-length / 2, length / 2), rnd.uniform(-width / 2, width / 2)
        x = cx + u * math.cos(a) - w * math.sin(a) + rnd.gauss(0, 0.05)
        y = cy + u * math.sin(a) + w * math.cos(a) + rnd.gauss(0, 0.05)
        X.append([round(x, 2), round(y, 2)])
    return X

# ---- метод k ближайших соседей (Б1.2) и кросс-валидация (Б2.1)
def dist(a, b):
    return math.sqrt(sum((ai - bi) ** 2 for ai, bi in zip(a, b)))

def knn_predict(X_train, y_train, X_test, k):
    result = []
    for x in X_test:
        pairs = sorted(((dist(xi, x), yi) for xi, yi in zip(X_train, y_train)), key=lambda p: p[0])
        labels = [yi for d, yi in pairs[:k]]
        counts = {}
        for lab in labels:
            counts[lab] = counts.get(lab, 0) + 1
        best = max(counts.values())
        result.append(next(lab for lab in labels if counts[lab] == best))   # ничья – решает ближний
    return result

def accuracy(y_true, y_pred):
    return sum(1 for a, b in zip(y_true, y_pred) if a == b) / len(y_true)

def k_fold_indices(n, n_folds, seed=RANDOM_STATE):
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    folds, start = [], 0
    for i in range(n_folds):
        size = n // n_folds + (1 if i < n % n_folds else 0)
        folds.append(idx[start:start + size])
        start += size
    return folds

def cross_val_score(model, X, y, folds, params):
    scores = []
    for i in range(len(folds)):
        tr = [j for f in range(len(folds)) if f != i for j in folds[f]]
        y_pred = model([X[j] for j in tr], [y[j] for j in tr], [X[j] for j in folds[i]], params)
        scores.append(accuracy([y[j] for j in folds[i]], y_pred))
    return sum(scores) / len(scores)

# ---- масштабирование (Б1.2, Б1.6): параметры – только по обучающей выборке
def minmax_fit(X):
    return [min(r[j] for r in X) for j in range(len(X[0]))], [max(r[j] for r in X) for j in range(len(X[0]))]

def minmax_transform(X, mins, maxs):
    return [[(v - lo) / (hi - lo) for v, lo, hi in zip(row, mins, maxs)] for row in X]

def standardize_fit(X):
    n = len(X)
    mu = [sum(r[j] for r in X) / n for j in range(len(X[0]))]
    sigma = [math.sqrt(sum((r[j] - mu[j]) ** 2 for r in X) / n) for j in range(len(X[0]))]
    return mu, sigma

def standardize_apply(X, mu, sigma):
    return [[(v - m) / s for v, m, s in zip(row, mu, sigma)] for row in X]

# Объекты на дороге, как в Б2.1: три полезных признака (скорость, длина, высота), нормализация по обучающей выборке.
X_road, y_road = make_road_objects()
Xr_tr_raw, Xr_te_raw, yr_train, yr_test = train_test_split(X_road, y_road)
mins, maxs = minmax_fit([r[:3] for r in Xr_tr_raw])
Xr_train = minmax_transform([r[:3] for r in Xr_tr_raw], mins, maxs)
Xr_test = minmax_transform([r[:3] for r in Xr_te_raw], mins, maxs)

# Набор Wine: 178 вин, 13 признаков, три сорта; разбиение как в Б2.1.
wine = load_wine()
X_wine, y_wine = wine.data.tolist(), wine.target.tolist()
FEATURES = wine.feature_names
Xw_train, Xw_test, yw_train, yw_test = train_test_split(X_wine, y_wine)

X_car = make_car_cloud()
print('дорога:', len(Xr_train), len(Xr_test), '  Wine:', len(Xw_train), len(Xw_test), '  машина:', len(X_car))
print('  ожидается: дорога 210 90, Wine 125 53, машина 200')
print('точность метода k ближайших соседей (k = 13) на дороге:', round(accuracy(yr_test, knn_predict(Xr_train, yr_train, Xr_test, 13)), 3), '  ожидается 0.911, как в Б2.1')

**Задание П1. Проклятие размерности на опыте.**

1. `random_cloud(n, d, rnd)` – $n$ случайных точек в кубе $[0, 1]^d$ (каждая координата – `rnd.random()`).
2. `near_far_ratio(X, x)` – отношение расстояния от точки `x` до ближайшей точки из `X` к расстоянию до самой далекой.
3. `shell_share(d, width)` – доля объема куба $[0, 1]^d$ в слое толщиной `width` у граней: $1 - (1 - 2 \cdot \text{width})^d$ (внутренний куб имеет сторону $1 - 2 \cdot \text{width}$).
4. Добавьте к трем полезным признакам объектов на дороге $m$ случайных признаков (`add_random_features`) и посмотрите, что станет с точностью метода k ближайших соседей.

*Считаем на бумаге.* На прямой точки 0,2 и 0,9, новая точка 0,3: ближайшая на расстоянии 0,1, самая далекая на расстоянии 0,6, отношение $0{,}1/0{,}6 \approx 0{,}167$.

In [ ]:
def random_cloud(n, d, rnd):
    return ...                                  # <-- n точек по d координат

def near_far_ratio(X, x):
    ds = ...                                    # <-- расстояния от x до всех точек
    return ...                                  # <-- ближайшее к самому далекому

def shell_share(d, width=0.05):
    return ...                                  # <-- доля объема у граней


print(round(near_far_ratio([[0.2], [0.9]], [0.3]), 3), '  ожидается 0.167')

rnd = random.Random(RANDOM_STATE)
dims = [1, 2, 3, 5, 10, 20, 50, 100, 200, 500, 1000]
ratios = []
for d in dims:
    X = random_cloud(200, d, rnd)
    r = ...                                     # <-- среднее по 20 новым точкам
    ratios.append(r)
print(' d     ближайшая / самая далекая     доля объема у граней')
for d, r in zip(dims, ratios):
    print(f'{d:5d}   {r:.3f}                         {shell_share(d):.3f}')
print('  ожидается: d = 1 – около 0.00;  d = 10 – около 0.34;  d = 1000 – около 0.91;  доля у граней: d = 2 – 0.190, d = 50 – 0.995')

In [ ]:
def add_random_features(X, m, rnd):
    return ...                                  # <-- m случайных признаков от 0 до 1


print(' m   точность на тесте')
noise_acc = {}
for m in [0, 1, 2, 5, 10, 20, 50]:
    rnd = random.Random(7)
    A = add_random_features(Xr_train, m, rnd)
    B = add_random_features(Xr_test, m, rnd)
    noise_acc[m] = ...                          # <-- метод k ближайших соседей, k = 13
    print(f'{m:2d}   {noise_acc[m]:.3f}')
print('  ожидается: 0 – 0.911;  2 – 0.878;  5 – 0.700;  10 – 0.633;  20 – 0.600;  50 – 0.433')

In [ ]:
# Готовая ячейка: графики
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].plot(dims, ratios, marker='o', color=BLUE)
ax[0].set_xscale('log'); ax[0].set_xlabel('размерность d'); ax[0].set_ylabel('ближайшая / самая далекая')
ax[0].set_title('Расстояния выравниваются'); ax[0].grid(alpha=0.3)
ax[1].plot(list(noise_acc), list(noise_acc.values()), marker='o', color=NAVY)
ax[1].axhline(1 / 3, color=GREY, linestyle='--', label='угадывание: 1/3')
ax[1].set_xlabel('число случайных признаков'); ax[1].set_ylabel('точность на тесте')
ax[1].set_title('Метод k ближайших соседей и лишние признаки'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Вопрос П1.** а) Почему при $d = 1000$ отношение близко к 1? Подсказка: квадрат расстояния – сумма $d$ независимых слагаемых. б) Почему 50 случайных признаков почти уничтожают метод k ближайших соседей, хотя три полезных признака никуда не делись? в) Лидар дает в секунду сотни тысяч точек, камера – миллионы пикселей. Почему модели для таких данных не работают с «сырыми» признаками напрямую?

*Ваш ответ:*

**Задание П2. Центрирование и ковариационная матрица.**

- `mean_vector(X)` – список средних по каждому признаку.
- `center(X, mu)` – вычесть из каждого объекта вектор средних.
- `covariance(X)` – ковариационная матрица по п. 4 теории (делим на $n$). Матрица симметрична, поэтому достаточно посчитать $c_{jl}$ при $j \le l$ и записать то же число в $c_{lj}$.

*Считаем на бумаге.* Точки $(1, 2), (2, 1), (3, 4), (6, 5)$. Средние $(3, 3)$; после центрирования $(-2, -1), (-1, -2), (0, 1), (3, 2)$. Дисперсия первого признака $(4 + 1 + 0 + 9)/4 = 3{,}5$, второго $(1 + 4 + 1 + 4)/4 = 2{,}5$, ковариация $(2 + 2 + 0 + 6)/4 = 2{,}5$.

Затем посчитаем ковариационную матрицу облака точек машины, повернутой на 30° (готовая функция `make_car_cloud`).

In [ ]:
def mean_vector(X):
    n = len(X)
    return ...                                  # <-- среднее каждого признака

def center(X, mu):
    return ...                                  # <-- вычесть средние

def covariance(X):
    Z = center(X, mean_vector(X))
    n, d = len(X), len(X[0])
    Cm = [[0.0] * d for _ in range(d)]
    for j in range(d):
        for l in range(j, d):
            Cm[j][l] = ...                      # <-- средняя сумма произведений отклонений
            Cm[l][j] = ...                      # <-- симметрия
    return Cm


T = [[1, 2], [2, 1], [3, 4], [6, 5]]
print(mean_vector(T), covariance(T), '  ожидается [3.0, 3.0] [[3.5, 2.5], [2.5, 2.5]]')
C_car = covariance(X_car)
print('среднее машины:', [round(v, 3) for v in mean_vector(X_car)], '  ожидается [10.033, 5.046]')
print('ковариационная матрица машины:', [[round(v, 3) for v in row] for row in C_car])
print('  ожидается [[1.322, 0.611], [0.611, 0.635]]')

In [ ]:
# Готовая ячейка: сверка с numpy (bias=True – деление на n) и облако точек
import numpy as np
print('наибольшее расхождение с numpy:', float(np.max(np.abs(np.array(C_car) - np.cov(np.array(X_car).T, bias=True)))), '  ожидается около 1e-15')
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.scatter([p[0] for p in X_car], [p[1] for p in X_car], s=10, color=BLUE)
ax.set_aspect('equal'); ax.grid(alpha=0.3); ax.set_xlabel('x, м'); ax.set_ylabel('y, м'); ax.set_title('Точки лидара на кузове машины')
plt.tight_layout(); plt.show()

**Вопрос П2.** а) Почему ковариация координат машины положительна? Каким был бы ее знак, если бы машина была повернута на −30°? А на 0°? б) Дисперсия по $x$ (1,322) больше, чем по $y$ (0,635). Значит ли это, что машина вытянута вдоль оси $x$? в) Что покажет ковариационная матрица, если машину измерять в сантиметрах вместо метров?

*Ваш ответ:*

**Задание П3. Степенной метод: первая главная компонента.**

- `matvec(A, v)` – произведение матрицы на вектор: элемент $i$ результата равен $\sum_j a_{ij} v_j$.
- `dot(a, b)` – скалярное произведение, `norm(v)` – длина вектора $\sqrt{v \cdot v}$.
- `power_iteration(Cm, n_iter=1000, tol=1e-10, seed)` по п. 8 теории:
  1. начальный вектор – случайные числа `rnd.gauss(0, 1)`, приведенные к длине 1;
  2. повторять: $w = C v / \|C v\|$; если все координаты $w$ отличаются от $v$ меньше чем на `tol`, остановиться;
  3. $\lambda = v^{\top} C v$; повернуть знак, чтобы наибольшая по модулю координата была положительной.

  Функция возвращает `(v, lam, iterations)`.

*Считаем на бумаге.* Для матрицы $\begin{pmatrix} 2 & 1 \\ 1 & 2 \end{pmatrix}$ вектор $(1, 1)$ – собственный: $C(1, 1) = (3, 3) = 3 \cdot (1, 1)$. После нормировки $(0{,}707;\ 0{,}707)$, $\lambda = 3$.

In [ ]:
def matvec(A, v):
    return ...                                  # <-- строка на вектор

def dot(a, b):
    return sum(x * y for x, y in zip(a, b))

def norm(v):
    return ...                                  # <-- длина вектора

def power_iteration(Cm, n_iter=1000, tol=1e-10, seed=RANDOM_STATE):
    rnd = random.Random(seed)
    v = [rnd.gauss(0, 1) for _ in Cm]
    v = ...                                     # <-- длина 1
    iterations = 0
    for iterations in range(1, n_iter + 1):
        w = ...                                 # <-- умножить на матрицу
        w = ...                                 # <-- снова длина 1
        done = max(abs(a - b) for a, b in zip(w, v)) < tol
        v = w
        if done:
            break
    lam = ...                                   # <-- дисперсия вдоль v
    j = max(range(len(v)), key=lambda i: abs(v[i]))
    if v[j] < 0:
        v = ...                                 # <-- наибольшая по модулю координата положительна
    return v, lam, iterations


v, lam, it = power_iteration([[2, 1], [1, 2]])
print([round(x, 4) for x in v], round(lam, 6), '  ожидается [0.7071, 0.7071] 3.0')
v1_car, lam1_car, it_car = power_iteration(C_car)
angle = ...                                     # <-- угол первой компоненты, градусы
print(f'машина: v1 = {[round(x, 3) for x in v1_car]}, дисперсия {lam1_car:.3f}, угол {angle:.1f}°, итераций {it_car}')
print('  ожидается: v1 = [0.863, 0.505], дисперсия 1.679, угол 30.3°')

**Вопрос П3.** а) Генератор поворачивал машину на 30°. Почему метод нашел 30,3°, а не ровно 30°? б) Как можно использовать первую компоненту в беспилотном автомобиле? в) Почему вектор $(-0{,}863;\ -0{,}505)$ – тоже правильный ответ, и зачем мы договорились о знаке?

*Ваш ответ:*

**Задание П4. Исчерпывание, все компоненты и проекция.**

- `deflate(Cm, lam, v)` – новая матрица $c'_{jl} = c_{jl} - \lambda v_j v_l$ (п. 9 теории).
- `pca_fit(X, n_components)` – обучение метода главных компонент: средние $\mu$, ковариационная матрица, затем `n_components` раз степенной метод и исчерпывание. Возвращает `mu`, список компонент, список их дисперсий $\lambda$ и общую дисперсию (сумму диагонали **исходной** матрицы).
- `pca_transform(X, mu, components)` – для каждого объекта список проекций $t_m = (x - \mu) \cdot v_m$ (п. 10 теории).

Проверим на машине: вторая компонента должна быть перпендикулярна первой, а разброс проекций вдоль компонент – дать длину и ширину кузова.

In [ ]:
def deflate(Cm, lam, v):
    d = len(Cm)
    return ...                                  # <-- вычесть найденную компоненту

def pca_fit(X, n_components):
    mu = mean_vector(X)
    Cm = covariance(X)
    total = ...                                 # <-- общая дисперсия: сумма диагонали
    components, variances = [], []
    for _ in range(n_components):
        v, lam, it = ...                        # <-- очередная компонента
        components.append(v)
        variances.append(lam)
        Cm = ...                                # <-- исчерпывание
    return mu, components, variances, total

def pca_transform(X, mu, components):
    return ...                                  # <-- проекции центрированного объекта


mu_car, comps_car, vars_car, total_car = pca_fit(X_car, 2)
print('компоненты:', [[round(x, 3) for x in v] for v in comps_car], '  ожидается [[0.863, 0.505], [-0.505, 0.863]]')
print('скалярное произведение v1 и v2:', round(dot(comps_car[0], comps_car[1]), 10), '  ожидается 0.0')
print('доли дисперсии:', [round(lam / total_car, 3) for lam in vars_car], '  ожидается [0.858, 0.142]')
T_car = pca_transform(X_car, mu_car, comps_car)
length = ...                                    # <-- размах проекций на первую компоненту
width = ...                                     # <-- размах проекций на вторую
print(f'оценка размеров кузова: {length:.2f} × {width:.2f} м   ожидается 4.62 × 1.89 (на самом деле 4.5 × 1.8)')

In [ ]:
# Готовая ячейка: компоненты на облаке и облако в новых координатах
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].scatter([p[0] for p in X_car], [p[1] for p in X_car], s=10, color=BLUE, alpha=0.6)
for v, lam, col in zip(comps_car, vars_car, (NAVY, DARK_BLUE)):
    s = 2 * math.sqrt(lam)
    ax[0].annotate('', xy=(mu_car[0] + s * v[0], mu_car[1] + s * v[1]), xytext=mu_car,
                   arrowprops=dict(arrowstyle='->', color=col, lw=2.5))
ax[0].set_aspect('equal'); ax[0].grid(alpha=0.3); ax[0].set_title('Главные компоненты: стрелки длиной 2σ')
ax[1].scatter([t[0] for t in T_car], [t[1] for t in T_car], s=10, color=BLUE, alpha=0.6)
ax[1].set_aspect('equal'); ax[1].grid(alpha=0.3); ax[1].set_xlabel('t1'); ax[1].set_ylabel('t2')
ax[1].set_title('Та же машина в координатах компонент')
plt.tight_layout(); plt.show()

**Вопрос П4.** а) Почему после исчерпывания степенной метод находит именно вторую компоненту, а не снова первую? б) В новых координатах машина стоит «ровно». Что сделал метод главных компонент с облаком точек геометрически? в) Сумма двух дисперсий $\lambda_1 + \lambda_2$ равна общей дисперсии. Почему?

*Ваш ответ:*

**Задание П5. Wine: 13 признаков на плоскости.**

1. Стандартизуйте все 178 вин и найдите все 13 компонент. Здесь мы только смотрим на данные и ничего не оцениваем, поэтому обучаем метод на всем наборе; при оценке модели (П7) так делать нельзя.
2. Посчитайте доли объясненной дисперсии и их накопленную сумму. Сколько компонент нужно, чтобы сохранить 90 % дисперсии?
3. Спроецируйте вина на две первые компоненты и нарисуйте их цветом сорта.
4. Посмотрите на веса первой компоненты: какие признаки входят в нее с наибольшими по модулю весами?
5. Повторите без стандартизации и сравните долю первой компоненты.

In [ ]:
mu_s, sigma_s = standardize_fit(X_wine)
Z_wine = standardize_apply(X_wine, mu_s, sigma_s)
mu_w, comps_w, vars_w, total_w = pca_fit(Z_wine, 13)
ratios_w = ...                                  # <-- доли объясненной дисперсии
cumulative, s = [], 0
for r in ratios_w:
    s += r
    ...                                         # <-- накопленная доля
k90 = ...                                       # <-- первое число компонент с долей не меньше 0.9
print('общая дисперсия:', round(total_w, 3), '  ожидается 13.0: у каждого стандартизованного признака дисперсия 1')
print('доли:', [round(r, 3) for r in ratios_w])
print('  ожидается [0.362, 0.192, 0.111, 0.071, 0.066, 0.049, 0.042, 0.027, 0.022, 0.019, 0.017, 0.013, 0.008]')
print('две компоненты:', round(cumulative[1], 3), ' компонент для 90 %:', k90, '  ожидается 0.554 и 8')

top = ...                                       # <-- четыре признака с наибольшим |весом|
print('первая компонента, главные веса:', [(FEATURES[j], round(comps_w[0][j], 3)) for j in top])
print("  ожидается flavanoids 0.423, total_phenols 0.395, od280/od315_of_diluted_wines 0.376, proanthocyanins 0.313")

mu_raw, comps_raw, vars_raw, total_raw = pca_fit(X_wine, 2)
print('без стандартизации: доля первой компоненты', round(vars_raw[0] / total_raw, 4),
      ' вес пролина', round(comps_raw[0][FEATURES.index('proline')], 4), '  ожидается 0.9981 и 0.9998')

In [ ]:
# Готовая ячейка: сверка со sklearn и рисунки
from sklearn.decomposition import PCA
sk = PCA().fit(np.array(Z_wine))
print('наибольшее расхождение долей со sklearn:', float(np.max(np.abs(np.array(ratios_w) - sk.explained_variance_ratio_))), '  ожидается около 1e-15')
print('|косинус| между нашими и sklearn компонентами:', [round(abs(float(np.dot(a, b))), 6) for a, b in zip(comps_w, sk.components_)][:5], '...  ожидается 1.0')
P_wine = pca_transform(Z_wine, mu_w, comps_w[:2])
P_raw = pca_transform(X_wine, mu_raw, comps_raw)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].bar(range(1, 14), ratios_w, color=BLUE, label='доля компоненты')
ax[0].plot(range(1, 14), cumulative, marker='o', color=NAVY, label='накопленная доля')
ax[0].axhline(0.9, color=GREY, linestyle='--'); ax[0].set_xlabel('номер компоненты'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[0].set_title('Доля объясненной дисперсии')
for c in range(3):
    ax[1].scatter([p[0] for p, y in zip(P_wine, y_wine) if y == c], [p[1] for p, y in zip(P_wine, y_wine) if y == c],
                  s=14, color=CLASS_COLORS[c], label=f'сорт {c}')
    ax[2].scatter([p[0] for p, y in zip(P_raw, y_wine) if y == c], [p[1] for p, y in zip(P_raw, y_wine) if y == c],
                  s=14, color=CLASS_COLORS[c], label=f'сорт {c}')
ax[1].set_title('Со стандартизацией: 55 % дисперсии'); ax[1].set_xlabel('t1'); ax[1].set_ylabel('t2'); ax[1].legend(); ax[1].grid(alpha=0.3)
ax[2].set_title('Без стандартизации: t1 – это пролин'); ax[2].set_xlabel('t1'); ax[2].legend(); ax[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Вопрос П5.** а) Метод главных компонент не видел сортов вина. Почему же на плоскости сорта разделились почти без перекрытия? б) Как прочитать первую компоненту по ее весам? в) Почему без стандартизации первая компонента объясняет 99,8 % дисперсии и почти совпадает с пролином? Хорошо ли это?

*Ваш ответ:*

**Задание П6. Восстановление и ошибка.**

- `pca_inverse(T, mu, components)` – восстановление $\hat{x} = \mu + t_1 v_1 + \dots + t_k v_k$ для каждого объекта (п. 10 теории).
- `reconstruction_error(X, R)` – средний по объектам квадрат расстояния между исходным объектом и восстановленным.

Проверим утверждение из теории: ошибка равна сумме отброшенных дисперсий.

In [ ]:
def pca_inverse(T, mu, components):
    result = []
    for t in T:
        x = list(mu)
        for tm, v in zip(t, components):
            x = ...                             # <-- прибавить t_m · v_m
        result.append(x)
    return result

def reconstruction_error(X, R):
    return ...                                  # <-- средний квадрат расстояния


T1 = pca_transform(X_car, mu_car, comps_car[:1])
R_car = pca_inverse(T1, mu_car, comps_car[:1])
print(f'машина, одна компонента: ошибка {reconstruction_error(X_car, R_car):.4f}, вторая дисперсия {vars_car[1]:.4f}   ожидается 0.2778 и 0.2778')

print(' k   ошибка    сумма отброшенных дисперсий')
for k in [1, 2, 5, 8, 13]:
    R = ...                                     # <-- сжать до k и восстановить
    print(f'{k:2d}   {reconstruction_error(Z_wine, R):.4f}    {sum(vars_w[k:]):.4f}')
print('  ожидается: 1 – 8.2941;  2 – 5.7972;  5 – 2.5789;  8 – 1.0377;  13 – 0.0000 в обоих столбцах')

In [ ]:
# Готовая ячейка: восстановление машины по одной компоненте
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.scatter([p[0] for p in X_car], [p[1] for p in X_car], s=10, color=GREY, alpha=0.5, label='исходные точки')
ax.scatter([p[0] for p in R_car], [p[1] for p in R_car], s=10, color=BLUE, label='по одной компоненте')
for p, r in list(zip(X_car, R_car))[:40]:
    ax.plot([p[0], r[0]], [p[1], r[1]], color=GREY, linewidth=0.6)
ax.set_aspect('equal'); ax.grid(alpha=0.3); ax.legend(fontsize=8); ax.set_title('Восстановление: проекции на прямую')
plt.tight_layout(); plt.show()

**Вопрос П6.** а) Куда попадают точки машины после сжатия до одной компоненты и восстановления? Чему равна длина серых отрезков? б) Объект Wine – 13 чисел. Если хранить только две проекции, во сколько раз меньше памяти нужно (не считая $\mu$ и компонент)? Какая доля дисперсии теряется? в) Почему на обучающей выборке ошибка восстановления точно равна сумме отброшенных дисперсий, а на новых данных – только примерно?

*Ваш ответ:*

**Задание П7. Метод k ближайших соседей после снижения размерности: честное сравнение.**

Модель по договору из Б2.1 – функция `model(X_train, y_train, X_val, params)`. Напишите `knn_pca_model`:

1. стандартизация: параметры `standardize_fit` – **только по `X_train`**, применить к обеим частям;
2. если `params['n_components']` не `None`: `pca_fit` **только по стандартизованной обучающей части**, затем `pca_transform` обеих частей;
3. метод k ближайших соседей с `params['k']`.

Сравним через 5-блочную кросс-валидацию на обучающей выборке Wine (125 вин) разное число компонент при $k = 5$, затем проверим лучший вариант на тесте. В конце посмотрим, что будет, если обучить метод главных компонент на всей обучающей выборке до кросс-валидации.

In [ ]:
def knn_pca_model(X_train, y_train, X_val, params):
    mu, sigma = ...                             # <-- параметры только по обучающей части
    A, B = standardize_apply(X_train, mu, sigma), standardize_apply(X_val, mu, sigma)
    if params['n_components'] is not None:
        m, comps, _, _ = ...                    # <-- компоненты только по обучающей части
        A, B = pca_transform(A, m, comps), pca_transform(B, m, comps)
    return ...                                  # <-- прогноз для валидационной части


folds_w = k_fold_indices(len(Xw_train), 5)
cv_pca = {}
print('компонент   точность на кросс-валидации')
for nc in [1, 2, 3, 5, 8, 13, None]:
    cv_pca[nc] = ...                            # <-- 5 блоков
    print(f'{str(nc):>9s}   {cv_pca[nc]:.3f}')
print('  ожидается: 1 – 0.848;  2 – 0.968;  3 – 0.984;  5 – 0.968;  8 – 0.960;  13 – 0.976;  None (без метода) – 0.976')

for nc in [2, None]:
    acc = accuracy(yw_test, knn_pca_model(Xw_train, yw_train, Xw_test, {'k': 5, 'n_components': nc}))
    print(f'тест, число компонент {nc}: {acc:.3f}')
print('  ожидается: 2 компоненты – 0.925;  без метода – 0.943')

In [ ]:
# Готовая ячейка: так делать нельзя – стандартизация и компоненты по всей обучающей выборке до кросс-валидации
mu_all, sigma_all = standardize_fit(Xw_train)
A_all = standardize_apply(Xw_train, mu_all, sigma_all)
m_all, comps_all, _, _ = pca_fit(A_all, 2)
P_all = pca_transform(A_all, m_all, comps_all)
leak = cross_val_score(lambda a, b, c, p: knn_predict(a, b, c, p['k']), P_all, yw_train, folds_w, {'k': 5})
print(f'с утечкой: {leak:.3f},  честно: {cv_pca[2]:.3f}   ожидается 0.976 и 0.968')

**Вопрос П7.** а) Две компоненты вместо 13 признаков: что выиграли и что потеряли? Почему одна компонента – слишком мало? б) Утечка здесь дала прибавку всего 0,008. Почему так мало, ведь в Б2.1 отбор признаков по всей выборке дал прибавку 0,4? И зачем тогда соблюдать правило? в) Беспилотный автомобиль встречает объекты по одному. Почему для него просто невозможно «обучить метод главных компонент на тестовых данных»?

*Ваш ответ:*

**Задание П8. Когда метод главных компонент не помогает.**

Вернемся к объектам на дороге из П1 и сравним два вида лишних признаков:

1. **20 случайных признаков** (функция `add_random_features` из П1): шум, никак не связанный с объектом;
2. **20 зашумленных копий** полезных признаков: признак $j$ копии – это полезный признак $j \bmod 3$ плюс случайная добавка `rnd.gauss(0, 0.1)` (как если бы несколько датчиков измеряли одно и то же).

В обоих случаях сравните метод k ближайших соседей ($k = 13$) на всех признаках и после метода главных компонент, обученного на обучающей выборке.

In [ ]:
def add_noisy_copies(X, m, rnd):
    return ...                                  # <-- копии полезных признаков с шумом

def knn_after_pca(A, B, n_components):
    m, comps, variances, total = ...            # <-- обучение только на A
    acc = accuracy(yr_test, knn_predict(pca_transform(A, m, comps), yr_train, pca_transform(B, m, comps), 13))
    return acc, [round(v / total, 3) for v in variances]


rnd = random.Random(7)
A_noise, B_noise = add_random_features(Xr_train, 20, rnd), add_random_features(Xr_test, 20, rnd)
rnd = random.Random(7)
A_copy, B_copy = add_noisy_copies(Xr_train, 20, rnd), add_noisy_copies(Xr_test, 20, rnd)

acc, shares = knn_after_pca(A_noise, B_noise, 3)
print(f'случайные признаки:  все 23 – {accuracy(yr_test, knn_predict(A_noise, yr_train, B_noise, 13)):.3f};  3 компоненты – {acc:.3f}, доли {shares}')
print('  ожидается: 0.600;  0.378, доли [0.074, 0.068, 0.064]')
acc, shares = knn_after_pca(A_copy, B_copy, 2)
print(f'зашумленные копии:   все 23 – {accuracy(yr_test, knn_predict(A_copy, yr_train, B_copy, 13)):.3f};  2 компоненты – {acc:.3f}, доли {shares}')
print('  ожидается: 0.922;  0.944, доли [0.518, 0.219]')
print('дисперсии полезных признаков:', [round(covariance(Xr_train)[j][j], 3) for j in range(3)], ' случайного признака: 1/12 ≈ 0.083')

**Вопрос П8.** а) Почему со случайными признаками метод главных компонент сделал только хуже (0,378 против 0,600)? Сравните дисперсии. б) Почему с зашумленными копиями он помог и даже превзошел исходные три признака? в) Сформулируйте правило: в каких данных метод главных компонент полезен перед классификацией, а в каких нет?

*Ваш ответ:*

---
### Задания повышенной сложности

1. **Рукописные цифры 8×8.** Загрузите набор `load_digits` из sklearn (1797 изображений по 64 пикселя). Найдите 40 компонент своим `pca_fit`, постройте накопленную долю дисперсии, нарисуйте первые 8 компонент как изображения 8×8 («собственные цифры»), восстановите несколько цифр по 2, 5, 10, 20 и 40 компонентам и спроецируйте все цифры на плоскость двух первых компонент.
2. **Повернутые прямоугольники на парковке.** Возьмите генератор `make_scene` из Б2.4, поверните каждый объект на свой случайный угол, найдите объекты DBSCAN, а для каждого кластера по главным компонентам постройте повернутый прямоугольник, который его охватывает. Сравните найденные углы и размеры с заданными.
3. **Скорость степенного метода.** Для матриц $\begin{pmatrix} 1 & 0 \\ 0 & r \end{pmatrix}$ при $r = 0{,}5;\ 0{,}9;\ 0{,}99$ посчитайте число итераций до сходимости и сравните с оценкой $\ln(\text{tol}) / \ln r$. Почему для некоторых компонент Wine степенному методу нужны сотни итераций?
4. **Кластеризация после снижения размерности.** Кластеризуйте стандартизованный Wine методом k средних (Б2.3) с $k = 3$ в 13 признаках и в 2 главных компонентах. Сравните скорректированный индекс Рэнда (Б2.4) и коэффициент силуэта. Почему силуэт на плоскости заметно выше?
5. **Цифры и метод k ближайших соседей.** Разделите цифры на обучающую и тестовую выборки, сравните точность и время метода k ближайших соседей на 64 пикселях и на 10, 20 и 30 главных компонентах.

In [ ]:
# Место для заданий повышенной сложности